# nb_01 — DimDate

**Purpose:** Generate the `DimDate` dimension table covering **2022-01-01 through 2024-12-31** (1 096 rows).

## Schema
| Column | Type | Description |
|---|---|---|
| date_key | INT | YYYYMMDD surrogate key |
| year | INT | Calendar year |
| quarter | INT | 1–4 |
| month | INT | 1–12 |
| day | INT | Day of month |
| weekday | INT | 0=Monday … 6=Sunday |
| is_weekend | BOOLEAN | True when weekday >= 5 |

Writes to Delta table `DimDate` (overwrite mode).

In [ ]:
from datetime import date, timedelta

rows = []
d = date(2022, 1, 1)
while d <= date(2024, 12, 31):
    rows.append((
        int(d.strftime("%Y%m%d")),
        d.year,
        (d.month - 1) // 3 + 1,
        d.month,
        d.day,
        d.weekday(),
        d.weekday() >= 5
    ))
    d += timedelta(1)

print(f"Total rows generated: {len(rows)}")

In [ ]:
schema = "date_key INT, year INT, quarter INT, month INT, day INT, weekday INT, is_weekend BOOLEAN"
df = spark.createDataFrame(rows, schema)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("DimDate")

print("DimDate written successfully.")
spark.sql("SELECT COUNT(*) AS row_count FROM DimDate").show()

In [ ]:
# Optimize the table for point-lookup access patterns
spark.sql("OPTIMIZE DimDate ZORDER BY (date_key)")
spark.sql("SELECT * FROM DimDate LIMIT 5").show()